In [1]:
import pandas as pd
import numpy as np

file_path = r"C:\Users\AGFirass\Documents\GitHub\Transformer-Based-DDoS-Detection\notebooks\feature_importance\correlation_analysis\reduced_dataset\reduced_dataset_correlation_filtered.parquet"
df = pd.read_parquet(file_path)

In [3]:
print("="*80)
print("DATASET OVERVIEW")
print("="*80)
print(f"Shape: {df.shape}")
print("\nColumns:\n", df.columns.tolist())
print("\nData Types:\n", df.dtypes)

DATASET OVERVIEW
Shape: (144000, 55)

Columns:
 ['Unnamed: 0', 'Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Bwd Packets/s', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'FIN Flag Count', 'SYN Flag Count', 'PSH Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'min_seg_size_forward', 'Active Mean', 'Active Std', 'Active Max

In [4]:
print("\nMissing values per column:\n", df.isna().sum())


Missing values per column:
 Unnamed: 0                        0
Flow ID                           0
Source IP                         0
Source Port                       0
Destination IP                    0
Destination Port                  0
Protocol                          0
Timestamp                         0
Flow Duration                     0
Total Fwd Packets                 0
Total Backward Packets            0
Total Length of Fwd Packets       0
Fwd Packet Length Std             0
Bwd Packet Length Max             0
Bwd Packet Length Min             0
Flow Bytes/s                   2477
Flow Packets/s                    0
Flow IAT Min                      0
Bwd IAT Total                     0
Bwd IAT Mean                      0
Bwd IAT Min                       0
Fwd PSH Flags                     0
Bwd PSH Flags                     0
Fwd URG Flags                     0
Bwd URG Flags                     0
Fwd Header Length                 0
Bwd Header Length                 0

In [7]:
print("\nChecking for infinite values...")
inf_cols = df.columns[(df == np.inf).any() | (df == -np.inf).any()]
print("Infinite values found in columns:", list(inf_cols))


Checking for infinite values...
Infinite values found in columns: ['Flow Bytes/s', 'Flow Packets/s']


In [8]:
df = df.replace([np.inf, -np.inf], np.nan)

In [11]:
# Fill NaNs with median (numeric cols only)
for col in df.select_dtypes(include=[np.number]).columns:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

# Fill NaNs in non-numeric columns with mode
for col in df.select_dtypes(exclude=[np.number]).columns:
    mode_val = df[col].mode()[0] if not df[col].mode().empty else None
    df[col] = df[col].fillna(mode_val)

# Missing values after
print("\nMissing values after handling:\n", df.isna().sum())


Missing values after handling:
 Unnamed: 0                     0
Flow ID                        0
Source IP                      0
Source Port                    0
Destination IP                 0
Destination Port               0
Protocol                       0
Timestamp                      0
Flow Duration                  0
Total Fwd Packets              0
Total Backward Packets         0
Total Length of Fwd Packets    0
Fwd Packet Length Std          0
Bwd Packet Length Max          0
Bwd Packet Length Min          0
Flow Bytes/s                   0
Flow Packets/s                 0
Flow IAT Min                   0
Bwd IAT Total                  0
Bwd IAT Mean                   0
Bwd IAT Min                    0
Fwd PSH Flags                  0
Bwd PSH Flags                  0
Fwd URG Flags                  0
Bwd URG Flags                  0
Fwd Header Length              0
Bwd Header Length              0
Bwd Packets/s                  0
Max Packet Length              0
Packet Len

In [12]:
df.shape

(144000, 55)

In [13]:
# 7. Check 'Label' column
print("\nLabel column unique values:")
print(df["Label"].unique())

if pd.api.types.is_numeric_dtype(df["Label"]):
    print("Label is numeric — possible label encoding.")
else:
    print("Label is not numeric — probably categorical.")


Label column unique values:
['BENIGN' 'DrDoS_DNS' 'DrDoS_LDAP' 'DrDoS_MSSQL' 'DrDoS_NetBIOS'
 'DrDoS_NTP' 'DrDoS_SNMP' 'DrDoS_SSDP' 'DrDoS_UDP' 'Syn' 'TFTP' 'UDP-lag']
Label is not numeric — probably categorical.


In [14]:
df = df.drop(columns=["Flow ID"], errors='ignore')

In [16]:
from sklearn.preprocessing import LabelEncoder

# Encode SimillarHTTP
if df["SimillarHTTP"].dtype == object:
    df["SimillarHTTP"] = df["SimillarHTTP"].fillna("Unknown")
    df["SimillarHTTP"] = LabelEncoder().fit_transform(df["SimillarHTTP"])

In [17]:
# Convert Timestamp to datetime
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors='coerce')

In [22]:
# Convert Timestamp to numeric (UNIX timestamp in seconds)
df["Timestamp"] = df["Timestamp"].astype("int64") // 10**9  # seconds

In [18]:
# Encode Label (0–11)
label_encoder = LabelEncoder()
df["Label"] = label_encoder.fit_transform(df["Label"])

In [19]:
print("\nLabel classes mapping:")
for idx, cls in enumerate(label_encoder.classes_):
    print(f"{cls} -> {idx}")


Label classes mapping:
BENIGN -> 0
DrDoS_DNS -> 1
DrDoS_LDAP -> 2
DrDoS_MSSQL -> 3
DrDoS_NTP -> 4
DrDoS_NetBIOS -> 5
DrDoS_SNMP -> 6
DrDoS_SSDP -> 7
DrDoS_UDP -> 8
Syn -> 9
TFTP -> 10
UDP-lag -> 11


In [21]:
# Encode Source IP and Destination IP into integer IDs
for col in ["Source IP", "Destination IP"]:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

In [23]:
# Check datatypes
print("\nData Types after processing:\n", df.dtypes)


Data Types after processing:
 Unnamed: 0                       int64
Source IP                        int64
Source Port                      int64
Destination IP                   int64
Destination Port                 int64
Protocol                         int64
Timestamp                        int64
Flow Duration                    int64
Total Fwd Packets                int64
Total Backward Packets           int64
Total Length of Fwd Packets    float64
Fwd Packet Length Std          float64
Bwd Packet Length Max          float64
Bwd Packet Length Min          float64
Flow Bytes/s                   float64
Flow Packets/s                 float64
Flow IAT Min                   float64
Bwd IAT Total                  float64
Bwd IAT Mean                   float64
Bwd IAT Min                    float64
Fwd PSH Flags                    int64
Bwd PSH Flags                    int64
Fwd URG Flags                    int64
Bwd URG Flags                    int64
Fwd Header Length                

In [24]:
print("\nFinal dataset shape:", df.shape)


Final dataset shape: (144000, 54)


In [25]:
df.to_parquet("processed_dataset_ready_for_transformer.parquet", index=False)